# Does a cortical neuron type's own supply of a regulator move its threshold?

Self-contained. Runtime → Run all. The download takes 10–20 minutes; the rest takes a few minutes.


In [ ]:
options(timeout = 7200)
for (p in c("data.table", "Matrix", "ggplot2")) if (!requireNamespace(p, quietly = TRUE)) install.packages(p)
suppressPackageStartupMessages({library(data.table); library(Matrix); library(ggplot2)})
sessionInfo()$R.version$version.string

## 1. Get the data
Downloads the deposited archive (about 2.9 GB) and extracts only the six files needed. If the download fails, upload `HD_cortex.zip` to the Files panel on the left and run this cell again.

In [ ]:
zip <- "HD_cortex.zip"
url <- "https://zenodo.org/records/18202664/files/HD_cortex.zip?download=1"
if (!file.exists(zip) || file.size(zip) < 2.5e9) {
  status <- system(paste("curl -L --fail -o", zip, shQuote(url)))
  if (status != 0) stop("Download failed. Upload HD_cortex.zip to the Files panel and rerun this cell.")
}
need <- c("HD_cortex/data/deepdive_dge_data/caudate/SPN/raw.dge.rds",
          "HD_cortex/data/deepdive_dge_data/caudate/SPN/summary.rds",
          "HD_cortex/data/deepdive_dge_data/cortex/glutamatergic/raw.dge.rds",
          "HD_cortex/data/deepdive_dge_data/cortex/glutamatergic/summary.rds",
          "HD_cortex/data/gene_expr_models/caudate.stats.txt",
          "HD_cortex/data/gene_expr_models/cortex.stats.txt")
system(paste("unzip -o", zip, paste(need, collapse = " ")))
file.exists(need)

## 2. Definitions
The candidate regulators, and the per-gene statistics used to pick the phase C genes.


In [ ]:
CANDIDATES <- c("MBNL1","MBNL2","CREBBP","EP300","EZH2","EED","SUZ12","TCERG1","MED15","HTT","MSH3")
GROUP_COLS <- c("subcluster_name", "subtype", "predClass_allen_gluta_subclass", "predClass_SPN_DI", "cluster_name", "predClass")

prep_region <- function(region, dir, stats_file, p_threshold = 1e-10, n_ref = 4000, seed = 1) {
  raw <- readRDS(file.path(dir, "raw.dge.rds"))
  sd  <- as.data.table(readRDS(file.path(dir, "summary.rds")))
  cat(region, ": summary columns:", paste(head(names(sd), 40), collapse = ", "), "\n")
  grp_col <- intersect(GROUP_COLS, names(sd))[1]
  cat("Neuron subtype column used:", grp_col, "\n")
  sd <- sd[!is.na(CAG_LENGTH) & CAG_LENGTH > 35 & CELL_BARCODE %in% colnames(raw)]
  sd[, GROUP := paste(DONOR, get(grp_col), sep = "|")]
  set.seed(seed)
  ref  <- sd[CAG_LENGTH <= 100]
  if (nrow(ref) > n_ref) ref <- ref[sample(.N, n_ref)]
  cells <- rbind(ref, sd[CAG_LENGTH > 100])
  st <- fread(stats_file)
  st[, SET := fifelse(PHASE_C_PVALUE < p_threshold, fifelse(PHASE_C_EFFECT < 0, "C-", "C+"), "other")]
  genes <- intersect(unique(c(st[SET != "other", GENE], CANDIDATES)), rownames(raw))
  N <- Matrix::colSums(raw[, cells$CELL_BARCODE])
  X <- as.matrix(raw[genes, cells$CELL_BARCODE])
  rm(raw); invisible(gc())
  cat(region, ":", nrow(cells), "nuclei (", sum(cells$CAG_LENGTH > 100), "with >100 CAGs ),", length(genes), "genes\n")
  list(region = region, cells = cells, N = N, X = X, st = st[GENE %in% genes])
}

## 3. Hinge model
Finds the repeat length at which a quantity starts to change, with a 95% interval.


In [ ]:
T_GRID <- seq(40, 300, by = 10)

expected_counts <- function(y, N, grp, is_ref) {
  rate <- tapply(y[is_ref], grp[is_ref], sum) / tapply(N[is_ref], grp[is_ref], sum)
  r <- rate[grp]; r[is.na(r) | r <= 0] <- sum(y[is_ref]) / sum(N[is_ref])
  pmax(as.numeric(N * r), 1e-8)
}

fit_hinge <- function(y, L, mu0) {
  off <- log(mu0)
  dev <- sapply(T_GRID, function(t) {
    h <- pmax(0, L - t) / 100
    f <- tryCatch(glm.fit(cbind(1, h), y, offset = off, family = poisson()), error = function(e) NULL)
    if (is.null(f) || !f$converged) NA_real_ else f$deviance
  })
  if (all(is.na(dev))) return(NULL)
  i <- which.min(dev); t_hat <- T_GRID[i]
  h <- pmax(0, L - t_hat) / 100
  f <- glm.fit(cbind(1, h), y, offset = off, family = poisson())
  phi <- max(1, sum((y - f$fitted.values)^2 / f$fitted.values) / f$df.residual)
  ok <- (dev - dev[i]) / phi <= qchisq(0.95, 1)
  f0 <- glm.fit(matrix(1, length(y), 1), y, offset = off, family = poisson())
  data.table(onset = t_hat, ci_low = min(T_GRID[which(ok)]), ci_high = max(T_GRID[which(ok)]),
             change_per_100 = exp(f$coefficients[2]),
             p_vs_no_change = pchisq((f0$deviance - dev[i]) / phi, 2, lower.tail = FALSE))
}

run_region <- function(R) {
  L <- R$cells$CAG_LENGTH; grp <- R$cells$GROUP; is_ref <- L <= 100
  res <- rbindlist(lapply(rownames(R$X), function(g) {
    y <- R$X[g, ]
    if (sum(y[is_ref]) < 20) return(NULL)
    out <- fit_hinge(y, L, expected_counts(y, R$N, grp, is_ref))
    if (is.null(out)) NULL else cbind(GENE = g, out)
  }))
  res <- merge(res, R$st[, .(GENE, SET, PHASE_C_PVALUE)], by = "GENE")
  res[, region := R$region]
  res
}

## 4. Does a neuron type's own supply of a regulator move its threshold?

If an expanded repeat works by binding up a limited supply of some regulator, a neuron type that makes more of that regulator should start changing at a longer repeat, and one that makes more HTT should start changing at a shorter repeat. This fits one threshold per cortical neuron subtype and compares it with that subtype's baseline level of each candidate, measured in its own nuclei with 36-100 CAGs.

In [ ]:
MIN_HIGH        <- 40     # nuclei above 100 CAGs a subtype needs to be fitted
REF_PER_SUBTYPE <- 1500   # reference nuclei kept per subtype

dir <- "HD_cortex/data/deepdive_dge_data/cortex/glutamatergic"
raw <- readRDS(file.path(dir, "raw.dge.rds"))
sdq <- as.data.table(readRDS(file.path(dir, "summary.rds")))
grp_col <- intersect(c("subcluster_name", "subtype", "predClass_allen_gluta_subclass"), names(sdq))[1]
cat("subtype column used:", grp_col, "\n")
sdq <- sdq[!is.na(CAG_LENGTH) & CAG_LENGTH > 35 & CELL_BARCODE %in% colnames(raw)]
sdq[, SUBTYPE := as.character(get(grp_col))]

st <- fread("HD_cortex/data/gene_expr_models/cortex.stats.txt")
st[, SET := fifelse(PHASE_C_PVALUE < 1e-10, fifelse(PHASE_C_EFFECT < 0, "C-", "C+"), "other")]
cplus  <- intersect(st[SET == "C+", GENE], rownames(raw))
cminus <- intersect(st[SET == "C-", GENE], rownames(raw))
cands  <- intersect(CANDIDATES, rownames(raw))

set.seed(1)
keep <- sdq[, {
  hi <- .SD[CAG_LENGTH > 100]; lo <- .SD[CAG_LENGTH <= 100]
  if (nrow(lo) > REF_PER_SUBTYPE) lo <- lo[sample(.N, REF_PER_SUBTYPE)]
  rbind(hi, lo)
}, by = SUBTYPE]

N  <- Matrix::colSums(raw[, keep$CELL_BARCODE])
Xp <- Matrix::colSums(raw[cplus,  keep$CELL_BARCODE])
Xm <- Matrix::colSums(raw[cminus, keep$CELL_BARCODE])
XC <- as.matrix(raw[cands, keep$CELL_BARCODE])
rm(raw); invisible(gc())
cat(nrow(keep), "nuclei,", length(unique(keep$SUBTYPE)), "subtypes,",
    length(cplus), "rising and", length(cminus), "falling phase C genes\n")

subs <- keep[, .(n = .N, n_high = sum(CAG_LENGTH > 100)), by = SUBTYPE][n_high >= MIN_HIGH]
cat(nrow(subs), "subtypes have at least", MIN_HIGH, "nuclei above 100 CAGs\n")

out <- rbindlist(lapply(subs$SUBTYPE, function(s) {
  i <- which(keep$SUBTYPE == s)
  L <- keep$CAG_LENGTH[i]; is_ref <- L <= 100; grp <- keep$DONOR[i]
  fp <- fit_hinge(Xp[i], L, expected_counts(Xp[i], N[i], grp, is_ref))
  fm <- fit_hinge(Xm[i], L, expected_counts(Xm[i], N[i], grp, is_ref))
  if (is.null(fp) || is.null(fm)) return(NULL)
  base <- rowSums(XC[, i[is_ref], drop = FALSE]) / sum(N[i[is_ref]]) * 1e5
  data.table(SUBTYPE = s, n = length(i), n_high = sum(!is_ref),
             median_UMIs = median(N[i]),
             onset_up = fp$onset, up_lo = fp$ci_low, up_hi = fp$ci_high, up_per_100 = fp$change_per_100,
             onset_dn = fm$onset, dn_lo = fm$ci_low, dn_hi = fm$ci_high, dn_per_100 = fm$change_per_100,
             as.list(base))
}))
fwrite(out, "subtype_thresholds.csv")
print(out[order(onset_up), .(SUBTYPE, n, n_high, onset_up, up_lo, up_hi, onset_dn, dn_lo, dn_hi)])

## 5. Is the threshold related to what the subtype makes?

Spearman correlation across subtypes between each subtype's threshold and its baseline level of each candidate, with the median number of RNA molecules per nucleus included as a check: a subtype that is simply read more deeply will look high on every gene at once.

In [ ]:
vars <- c(intersect(CANDIDATES, names(out)), "median_UMIs")
cors <- rbindlist(lapply(vars, function(g) rbindlist(lapply(c("onset_up", "onset_dn"), function(o) {
  x <- out[[g]]; y <- out[[o]]
  if (length(unique(x)) < 3 || length(unique(y)) < 3) return(NULL)
  ct <- suppressWarnings(cor.test(x, y, method = "spearman"))
  data.table(variable = g, threshold = o, rho = unname(ct$estimate), p = ct$p.value)
}))))
print(cors[order(p)])
cat("\nsubtypes compared:", nrow(out), "- a correlation needs to be strong to show up in so few\n")

options(repr.plot.width = 14, repr.plot.height = 8)
long <- melt(out, id.vars = c("SUBTYPE", "onset_up"), measure.vars = vars,
             variable.name = "gene", value.name = "baseline")
ggplot(long, aes(baseline, onset_up)) +
  geom_point(colour = "#2f6db5", size = 2) +
  geom_smooth(method = "lm", se = FALSE, colour = "#b9b4a8", linewidth = 0.6) +
  facet_wrap(~ gene, scales = "free_x") +
  labs(x = "Baseline expression in that subtype's 36-100 CAG nuclei (per 100,000 RNA molecules)",
       y = "CAG length at which that subtype's rising genes start to change") +
  theme_minimal(base_size = 12)

## 6. Download the results

`subtype_thresholds.csv` holds one row per cortical neuron subtype.